## Open Table Dress Code Finder


In [ ]:
%%capture output
%%shell
# Ubuntu no longer distributes chromium-browser outside of snap
#
# Proposed solution: https://askubuntu.com/questions/1204571/how-to-install-chromium-without-snap

# Add debian buster
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

# Add keys
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg

# Prefer debian repo for chromium* packages only
# Note the double-blank lines between entries
cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500


Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300


Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

In [ ]:
import requests
from lxml import html
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [ ]:
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36")


service = webdriver.chrome.service.Service("/usr/bin/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

# hard coded latitude and longitude!
restaurant_name = 'Le Jardinier'
# Formatted name is always just the name but instead of spaces it is %20
formatted_name = restaurant_name.replace(' ', '%20')

driver.get(f'https://www.opentable.com/s?latitude=40.74752&longitude=-73.98637&term={formatted_name}')

page_source = driver.page_source
driver.quit()

In [ ]:
from IPython.core.display import HTML
HTML(page_source)
# lxml thing (like beautifulsoup)
tree = html.fromstring(page_source)
link_to_the_restaurant_page = tree.xpath('(//div[@data-test="restaurant-cards"]//a)[1]/@href')
print(link_to_the_restaurant_page)

[]


In [ ]:
headers = {
    'accept-language': 'en-US,en;q=0.9',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'
}

In [ ]:
resp = requests.get(link_to_the_restaurant_page[0], headers=headers)
tree = html.fromstring(resp.text)
HTML(resp.text)

IndexError: ignored

In [ ]:
dress_code = tree.xpath('''
    (//*[contains(translate(normalize-space(text()), "QWERTYUIOPASDFGHJKLZXCVBNM", "qwertyuiopasdfghjklzxcvbnm"), "dress code")]/
    following-sibling::*[normalize-space(text())])[1]/text()''')
dress_code[0]

IndexError: ignored

## one method to output dress code

In [ ]:
def get_dress_code(restaurant_name):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36")

    service = webdriver.chrome.service.Service("/usr/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=options)

    formatted_name = restaurant_name.replace(' ', '%20')

    driver.get(f'https://www.opentable.com/s?latitude=40.74752&longitude=-73.98637&term={formatted_name}')

    page_source = driver.page_source
    driver.quit()

    tree = html.fromstring(page_source)
    link_to_the_restaurant_page = tree.xpath('(//div[@data-test="restaurant-cards"]//a)[1]/@href')

    headers = {
        'accept-language': 'en-US,en;q=0.9',
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'
    }

    resp = requests.get(link_to_the_restaurant_page[0], headers=headers)
    tree = html.fromstring(resp.text)

    dress_code = tree.xpath('''
        (//*[contains(translate(normalize-space(text()), "QWERTYUIOPASDFGHJKLZXCVBNM", "qwertyuiopasdfghjklzxcvbnm"), "dress code")]/
        following-sibling::*[normalize-space(text())])[1]/text()''')
    return dress_code[0]

In [ ]:
get_dress_code("STK - Rooftop")


## FashionCLIP model and Text Search


### import libraries and pacakges

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from keras.preprocessing import image
from PIL import Image
from random import sample
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as resnet_preprocess
from tensorflow.keras.applications.xception import Xception, preprocess_input as xception_preprocess
from tensorflow.keras.models import Model
from pathlib import Path
Image.LOAD_TRUNCATED_IMAGES = True
import requests
from io import BytesIO
import os
import shutil

In [ ]:
#download fashionClip Model
#run one minute
%%capture
!git clone https://github.com/patrickjohncyh/fashion-clip
!pip -q install transformers
!pip -q install git+https://github.com/openai/CLIP.git@c0065a27ad170a67d9572f693292b594982ab9b2
!pip -q install appdirs
!pip -q install boto3
!pip -q install annoy millify datasets
!pip -q install validators
!pip -q install umap-learn
!pip -q install matplotlib datashader bokeh holoviews scikit-image
import sys
sys.path.append("fashion-clip/")
from fashion_clip.fashion_clip import FashionCLIP
import pandas as pd
import numpy as np
from collections import Counter
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.linear_model import LogisticRegression


AttributeError: ignored

In [ ]:
%%capture
fclip = FashionCLIP('fashion-clip')

NameError: ignored

In [ ]:
#!pip install kaggle

In [ ]:
# #Make a directory named kaggle and copy the kaggle.json file there.
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# # change the permission of the file
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#!kaggle datasets download -d alvations/farfetch-listings

### import original data from drive

In [ ]:
!gdown  https://drive.google.com/uc?id=1gs1OeyXQvhA3RACg8TAFPAOgo-eTJzot


Downloading...
From: https://drive.google.com/uc?id=1gs1OeyXQvhA3RACg8TAFPAOgo-eTJzot
To: /content/farfetch.zip
100% 3.81G/3.81G [00:40<00:00, 93.8MB/s]


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
#run two minutes
from zipfile import ZipFile
file_name = '/content/farfetch.zip' #the file is your dataset exact name
with ZipFile(file_name, 'r') as zip:
  zip.extractall()
  print('Done')

Done


In [ ]:
data = pd.read_csv("/content/current_farfetch_listings.csv")
data.sample(5)

demo=data.copy()
demo.sample(5)

,Unnamed: 0,availableSizes,brand.id,brand.name,gender,hasSimilarProducts,id,images.cutOut,images.model,isCustomizable,...,priceInfo.currencyCode,priceInfo.discountLabel,priceInfo.finalPrice,priceInfo.formattedFinalPrice,priceInfo.formattedInitialPrice,priceInfo.initialPrice,priceInfo.installmentsLabel,priceInfo.isOnSale,shortDescription,stockTotal
94091,131,"[{'scaleId': 0, 'size': 'XXS'}, {'scaleId': 0,...",4224,Marni,women,True,13706699,https://cdn-images.farfetch-contents.com/13/70...,https://cdn-images.farfetch-contents.com/13/70...,False,...,SGD,NaN,1099,"$1,099","$1,099",1099,NaN,False,zip up hoodie,7
79489,109,"[{'scaleId': 0, 'size': 'One Size'}]",13926,Red Valentino,women,True,13804277,https://cdn-images.farfetch-contents.com/13/80...,https://cdn-images.farfetch-contents.com/13/80...,False,...,SGD,NaN,1460,"$1,460","$1,460",1460,NaN,False,RED(V) flower appliqué shoulder bag,2
13456,136,"[{'scaleId': 0, 'size': 'One Size'}]",2765,Alexander McQueen,women,False,14176024,https://cdn-images.farfetch-contents.com/14/17...,https://cdn-images.farfetch-contents.com/14/17...,False,...,SGD,NaN,270,$270,$270,270,NaN,False,skull envelope card holder,3
32302,82,"[{'scaleId': 0, 'size': 'XS'}, {'scaleId': 0, ...",506282,Clube Bossa,women,True,14074278,https://cdn-images.farfetch-contents.com/14/07...,https://cdn-images.farfetch-contents.com/14/07...,False,...,SGD,NaN,183,$183,$183,183,NaN,False,Wolfe bikini top,18
145542,102,"[{'scaleId': 0, 'size': 'XS'}]",3661813,Moscot,unisex,False,13323706,https://cdn-images.farfetch-contents.com/13/32...,https://cdn-images.farfetch-contents.com/13/32...,False,...,SGD,NaN,367,$367,$367,367,NaN,False,Mangito tortoise glasses,1


### save data into drive as csv

In [ ]:
# data.to_csv('data.csv', index=False)

In [ ]:
# source_path = "/content/data.csv"
# destination_path = "/content/drive/MyDrive/MODELS/dataset/data.csv"

# shutil.copy(source_path, destination_path)

### **import demo copy from drive and demo pre-processing**

In [ ]:
!gdown https://drive.google.com/uc?id=1PRsTD5R_xkLrgPtFHxy2qhLakE6wlGfc

In [ ]:
demo = pd.read_csv('/content/current_farfetch_listings.csv').copy()


In [ ]:
#drop duplicates for demo
demo = demo.drop(demo.columns[[0, 1]], axis=1)
demo.drop('merchandiseLabel', axis=1, inplace=True)
demo.drop('priceInfo.discountLabel', axis=1, inplace=True)
demo.drop('priceInfo.installmentsLabel', axis=1, inplace=True)

demo.isnull().sum()

In [ ]:
demo=demo.drop_duplicates('id')
len(demo)

In [ ]:
demo=demo[['brand.id','brand.name','gender','hasSimilarProducts','id','images.cutOut','shortDescription']]
demo.head(10)

In [ ]:
images=demo['images.cutOut'].tolist()
texts=demo['shortDescription'].tolist()

In [ ]:
def convert_url(url: str) -> str:
    # Split the URL and get the last part of the path
    file_name = url.split('/')[-1]

    # Combine the directory structure with the file name
    new_path = f'/content/cutout-img/cutout/{file_name}'

    return new_path

# Convert the URLs in the list
new_images = [convert_url(url) for url in images]
new_images[:10]

###**save images and texts into drive as pkl**


**we need new_images for correct formats**



In [ ]:
!gdown https://drive.google.com/uc?id=1-BLVY9h-b0FrJbzog3g2rZ8Cfg4NJvlY #images1.pkl

In [ ]:
!gdown https://drive.google.com/uc?id=1-BFsUGjjgob5ntXCc5zEyH8tnwnICFZB #text1.pkl

In [ ]:
import pickle

with open('images1.pkl', 'wb') as f:
    pickle.dump(new_images, f)

with open('texts1.pkl', 'wb') as f:
    pickle.dump(texts, f)


In [ ]:
# source_path_images = "/content/images1.pkl"
# destination_path_images = "/content/images1.pkl"
# shutil.copy(source_path_images, destination_path_images)

# source_path_texts = "/content/texts1.pkl"
# destination_path_texts = "/content/texts1.pkl"
# shutil.copy(source_path_texts, destination_path_texts)

### **import image and text pkl from drive**



In [ ]:
with open('/content/images1.pkl', 'rb') as f:
    images1 = pickle.load(f)

with open('/content/texts1.pkl', 'rb') as f:
    texts1 = pickle.load(f)


### save image and text embeddings array with shape (187616, 512) into drive


In [ ]:
# # we create image embeddings and text embeddings
# image_embeddings = fclip.encode_images(images1, batch_size=32)
# text_embeddings = fclip.encode_text(texts1, batch_size=32)

# # we normalize the embeddings to unit norm (so that we can use dot product instead of cosine similarity to do comparisons)
# image_embeddings = image_embeddings/np.linalg.norm(image_embeddings, ord=2, axis=-1, keepdims=True)
# text_embeddings = text_embeddings/np.linalg.norm(text_embeddings, ord=2, axis=-1, keepdims=True)

In [ ]:
# np.save('image_embeddings1.npy', image_embeddings)
# np.save('text_embeddings1.npy', text_embeddings)

# source_path = "/content/image_embeddings1.npy"
# destination_path = "/content/drive/MyDrive/MODELS/dataset/image_embeddings1.npy"

# shutil.copy(source_path, destination_path)
# source_path = "/content/text_embeddings1.npy"
# destination_path = "/content/drive/MyDrive/MODELS/dataset/text_embeddings1.npy"

# shutil.copy(source_path, destination_path)

#image_embeddings = np.load('/content/drive/MyDrive/destination_folder/image_embeddings.npy')

### **import image and text embeddings from drive**

In [ ]:
!gdown https://drive.google.com/uc?id=1-Er6PCN9BqqgKcbz71pyTnoDxSL7zegq #image_embeddings1

In [ ]:
!gdown https://drive.google.com/uc?id=1-QDaJu8l0VGBJEOClkOL4cuV_sDDuKbD #text_embeddings1

In [ ]:
image_embeddings = np.load('/content/image_embeddings1.npy')
texts_embeddings = np.load('/content/text_embeddings1.npy')


### cutout & model images display

In [ ]:
# Extracting the Image
def load_images():

    # Store the directory path in a varaible
    cutout_img_dir = "/content/cutout-img/cutout"
    model_img_dir = "/content/model-img/model"

    # list the images in these directories
    cutout_images = os.listdir(cutout_img_dir)
    model_images = os.listdir(model_img_dir)

    # load 10 Random Cutout Images: Sample out 10 images randomly from the above list
    sample_cutout_images = sample(cutout_images,10)
    fig = plt.figure(figsize=(10, 5))

    print("==============Cutout Images==============")
    for i, img_name in enumerate(sample_cutout_images):
        plt.subplot(2, 5, i+1)
        img_path = os.path.join(cutout_img_dir, img_name)
        loaded_img = image.load_img(img_path)
        img_array = image.img_to_array(loaded_img, dtype='int')
        plt.imshow(img_array)
        plt.axis('off')

    plt.show()
    print()
    # load 10 Random Model Images: Sample out 10 images randomly from the above list
    sample_model_images = sample(model_images,10)
    fig = plt.figure(figsize=(10,5))

    print("==============Model Images==============")
    for i, img_name in enumerate(sample_model_images):
        plt.subplot(2, 5, i+1)
        img_path = os.path.join(model_img_dir, img_name)
        loaded_img = image.load_img(img_path)
        img_array = image.img_to_array(loaded_img, dtype='int')
        plt.imshow(img_array)
        plt.axis('off')

    plt.show()

In [ ]:
load_images()


In [ ]:
# Join the images with path and add in the dataframe

# Store the directory path in a varaible
cutout_img_dir = "/content/cutout-img/cutout"
model_img_dir = "/content/model-img/model"

# list the directories
cutout_images = os.listdir(cutout_img_dir)
model_images = os.listdir(model_img_dir)

### **Product Search Engine based on Text**





In [ ]:
def find_corresponding_image(found_object_id, images_list):
    """
    Find the corresponding image in the images_list based on the found_object_id.

    :param found_object_id: ID of the found object
    :param images_list: List of image paths
    :return: Image path if a matching image is found, otherwise None
    """
    # Iterate through the image paths in the list
    for image_path in images_list:
        # Check if the image_path contains the found_object's ID
        if str(found_object_id) in image_path:
            # Return the matching image path
            return image_path
    # If no matching image is found, return None
    return None

In [ ]:
text_embedding = fclip.encode_text(["white sandals"], 32)[0]

id_of_matched_object = np.argmax(text_embedding.dot(image_embeddings.T))
found_object = demo['id'].iloc[id_of_matched_object].tolist()


display cutout image

In [ ]:
image_path = find_corresponding_image(found_object, images1)
Image.open(image_path)


In [ ]:
def find_image_path(text):
    text_embedding = fclip.encode_text([text], 32)[0]
    id_of_matched_object = np.argmax(text_embedding.dot(image_embeddings.T))
    found_object = demo['id'].iloc[id_of_matched_object].tolist()
    image_path = find_corresponding_image(found_object, images1)
    return image_path


In [ ]:
s=find_image_path('shirt with bird')
s#='/content/cutout-img/cutout/13727260_16732739_300.jpg'

In [ ]:
Image.open(s)

In [ ]:
def text_image_transfer(text):
  text_embedding = fclip.encode_text([text], 32)[0]
  id_of_matched_object = np.argmax(text_embedding.dot(image_embeddings.T))
  found_object = demo['id'].iloc[id_of_matched_object].tolist()
  image_path = find_corresponding_image(found_object, images1)
  display_image=Image.open(image_path)
  display_image.show()

**Engine Method**

In [ ]:
text_image_transfer("White Solid Slim Dress shoes")

## Dress Recommender

dress code option

In [ ]:
def casual_dress():
    return {
        'Men': ['T-shirt', 'Jeans', 'Sneakers'],
        'Women': ['Blouse', 'Skirt', 'Sandals']
    }

def smart_casual():
    return {
        'Men': ['Collared shirt', 'Chinos', 'Loafers'],
        'Women': ['Blouse', 'Tailored pants', 'Flats']
    }

def business_casual():
    return {
        'Men': ['Button-down shirt', 'Khakis', 'Dress shoes'],
        'Women': ['Dress', 'Blazer', 'Heels']
    }

def jacket_required():
    return {
        'Men': ['Blazer', 'Dress pants', 'Dress shoes'],
        'Women': ['Elegant dress', 'Sophisticated separates', 'Heels']
    }

def get_garment(dress_code):
    if dress_code == "Casual Dress":
        return casual_dress()
    elif dress_code == "Smart Casual":
        return smart_casual()
    elif dress_code == "Business Casual":
        return business_casual()
    elif dress_code == "Jacket Required":
        return jacket_required()
    else:
        return "Invalid dress code."

user input

In [ ]:
def customize_garment(garment):
    features = {
        'Color': ['Red', 'Blue', 'Green', 'Black', 'White'],
        'Pattern': ['Solid', 'Stripes', 'Plaid', 'Floral', 'Polka Dots'],
        'Fit': ['Slim', 'Regular', 'Relaxed', 'Tailored', 'Loose']
    }

    customized_garment = garment
    for feature, options in features.items():
        print(f"Choose {feature} for your {garment}:")
        for i, option in enumerate(options, start=1):
            print(f"{i}. {option}")

        choice = int(input("Enter the number of your choice: "))
        selected_option = options[choice - 1]
        customized_garment += f" ({selected_option})"
        print(f"You chose {selected_option} for {garment}.\n")

    return customized_garment

def image_recoommend_display(customized_women_garments):
  for i in customized_women_garments:
    text_image_transfer(i)

## Demo

In [ ]:
restaurant_name = 'Bowery Road'
dress_code=get_dress_code(restaurant_name)
dress_code

In [ ]:
garments = get_garment(dress_code)
customized_men_garments = []
customized_women_garments = []

# print("Men's options:")
# for garment in garments["Men"]:
#     customized_garment = customize_garment(garment)
#     customized_men_garments.append(customized_garment)
#     print(f"Your customized {garment}: {customized_garment}\n")

print("\nWomen's options:")
for garment in garments["Women"]:
    customized_garment = customize_garment(garment)
    customized_women_garments.append(customized_garment)
    #print(f"Your customized {garment}: {customized_garment}\n")
# print("Customized Men's Garments:", customized_men_garments)
print("Customized Women's Garments:", ', '.join(customized_women_garments))


In [ ]:
image_recoommend_display(customized_women_garments)

# **Front End**


## libraries and packages

In [ ]:
!pip install flask
!pip install -q flask-ngrok
!pip install Jinja2

In [ ]:
!pip install flask-ngrok --upgrade

In [ ]:
#Please do not run this notebook without changing the authtoken and putting your own
!pip install -q pyngrok
!ngrok authtoken 2P7b0SajbVOsvErRu0SBswizXcQ_hKXeNwwaVjH4oc5zwnoL  ##use your personal key plz

In [ ]:
from flask import Flask, render_template, request

## HTML

In [ ]:
import os

# Create a directory for the Flask app and templates
if not os.path.exists('flask_app'):
    os.mkdir('flask_app')
    os.mkdir('flask_app/templates')

### Dress Code Page

In [ ]:
dress_code_html='''
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <meta http-equiv="x-ua-compatible" content="ie=edge">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="generator" content="Gatsby 2.0.19">
  <link rel="stylesheet" href="bootstrap/css/bootstrap.min.css">

  <title>Restaurant Dress Code</title>


    <style>

 	  h1 {
 	  background-color: rgba(0, 0, 0, 0.7);
 	  color: white;
 	  font-size: 45px;
 	  margin-top: 50px;
 	  }

      form.search-form {
        display: flex;
        flex-direction: row;
        align-items: center;
      }

      label {
        display: inline-block;
        width: 120px; /* Set a fixed width for the label to align the input field and button */
      }

      input[type="text"],
      button {
        display: inline-block;
        vertical-align: middle;
      }

      .btn-dress-code {
        background-color: #007bff;
        color: #fff;
        border: none;
        padding: 0.5rem 1rem;
        margin-left: 1rem;
        cursor: pointer;
      }

      .wrapper {
 	    display: flex;
 	    flex-direction: column;
  		justify-content: center;
 		align-items: center;
 		height: 100vh; /* Set the height to full viewport height */
	  }

	  .wrapper h1 {
      text-align: center;
      }

      .form-div {
  	  display: flex;
  	  flex-wrap: wrap;
  	  justify-content: center;
   	  }

	  .tt-input {
  	  width: 700px;
  	  height: 40px;
  	  padding: 10px;
  	  border: 1px solid #ccc;
  	  border-radius: 5px;
 	  font-size: 20px;
 	  background-color: white !important;
	  }

	  input[type="submit"] {
  	  background-color: blue;
  	  border: none;
  	  color: white;
  	  padding: 15px 30px;
  	  border-radius: 5px;
  	  font-size: 20px;
 	  cursor: pointer;
	  }

	  input[type="submit"]:hover {
 	  background-color: #0069d9;
	  }

	  body {
	  background: linear-gradient(rgba(255, 255, 255, 0.2), rgba(255, 255, 255, 0.2)), url('https://res.cloudinary.com/the-infatuation/image/upload/c_fill,w_1200,ar_4:3,g_auto,f_auto/images/NYC_LaMercerie_Interior_EmilySchindler_010_nvia5w');
/*   	  background-image: url('https://res.cloudinary.com/the-infatuation/image/upload/c_fill,w_1200,ar_4:3,g_auto,f_auto/images/NYC_LaMercerie_Interior_EmilySchindler_010_nvia5w'); */
 	  background-size: cover;

	  }

    </style>

</head>
<body>


  <script src="https://cdnjs.cloudflare.com/ajax/libs/jquery/3.3.1/jquery.min.js" type="text/javascript"></script>
  <script src="js/bootstrap.min.js"></script>




  <div class="wrapper">

  <h1>Hello! What restaurant are you looking to dress up for today?</h1>

  <form class="form-div w-100 d-flex w-100" action="/" method="POST">

    <span class="twitter-typeahead" style="position: relative; display: inline-block;">
  	  <input type="text" name="restaurant" id="search_home" required placeholder="Enter restaurant name" class="w-100 tt-input" style="vertical-align: top; background-color: transparent; margin-left: 5px; width: 1000px;" autocomplete="off" spellcheck="false" dir="auto">
      <pre aria-hidden="true" style="position: absolute; visibility: hidden; white-space: pre; font-family: manrope; font-size: 17px; font-style: normal; font-variant: normal; font-weight: 400; word-spacing: 0px; letter-spacing: 0px; text-indent: 0px; text-rendering: auto; text-transform: none;"></pre>
      <div class="tt-menu" style="position: absolute; top: 100%; left: 0px; z-index: 100; display: none;"><div class="tt-dataset tt-dataset-suggestions"></div></div>
    </span>
    <input type="submit" value="Get Dress Code" style="margin-left: 5px">
  </form>

  </div>

</body>
</html>
'''

# <!DOCTYPE html>
# <html>
# <head>
#   <title>Restaurant Dress Code</title>
# </head>
# <body>
#   <h1>Enter Restaurant Name</h1>
#   <form action="/" method="POST">
#     <label for="restaurant_name">Restaurant Name:</label>
#     <input type="text" id="restaurant_name" name="restaurant_name" required>
#     <input type="submit" value="Get Dress Code">
#   </form>

#   {% if restaurant_name and dress_code %}
#   <div>
#     <h2>{{ restaurant_name }}'s Dress Code:</h2>
#     <p>{{ dress_code }}</p>
#   </div>
#   {% endif %}
# </body>
# </html>
# '''

### Selection Page

In [ ]:
index_html='''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <!-- Add custom CSS styles -->
    <style>
        body {
            font-size: 1.5rem; /* Make fonts larger */
            background-color: #F6F6F6!important; /* Set the same background color as the result page */
        }
        .container {
            max-width: 800px; /* Limit container width */
            margin: 0 auto; /* Center container horizontally */
            display: flex;
        	flex-direction: column;
        	justify-content: center;
        	min-height: 100vh;

        }
        .garment-option {
            background-color: #D9CAB3!important; /* Set the same card background color */
        	color: white; Set the text color to white
        	border-radius: 5px; Add rounded corners to cards
        	padding: 1rem; Add some padding inside cards
        	margin-bottom: 1.5rem; /* Add spacing between garment options */
        }
        form {
            padding-top: 1.5rem;
            padding-bottom: 1.5rem;
        }
        .garment-grid {
        	display: grid;
        	grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
        	gap: 1rem;
        	background-color: #D9CAB3!important;
    	}

/*
    	.card {
        	background-color: #6D9886!important;
        	padding: 1rem;
        	border-radius: 0.25rem;
    	}
 */

    	.btn-primary {
        	background-color: #6D9886!important;
        	border-color: 212121!important;
    	}
    	.center-btn {
    		display: flex;
    		justify-content: center;
		}
		.center-container {
    		display: flex;
    		justify-content: center;
    		align-items: center;
    		min-height: 20vh; /* Adjust this value according to your needs */
		}
		span {
        font-size: 2rem; /* Adjust the font size */
        margin-bottom: 5rem; /* Add space below the span */
    	}
    	h1, h2 {
        margin-bottom: 0.5rem; /* Reduce the space below the headers */
    	}

    </style>
    <link href="https://stackpath.bootstrapcdn.com/bootstrap/4.3.1/css/bootstrap.min.css" rel="stylesheet">
    <script src="https://code.jquery.com/jquery-3.6.0.min.js" integrity="sha384-KyZXEAg3QhqLMpG8r+Knujsl5/WM6em9/evhFwiImE/7f1uZ04M8ez7JrLzOvx+j" crossorigin="anonymous"></script>
	<script src="https://cdn.jsdelivr.net/npm/@popperjs/core@2.11.6/dist/umd/popper.min.js" integrity="sha384-oBqDVmMz4fnFO9gybBud6N5z6M5b6XKU2v+9D7yrKz1zp0NlQSc2Ho8QKjTb+ZmU" crossorigin="anonymous"></script>
	<script src="https://stackpath.bootstrapcdn.com/bootstrap/4.3.1/js/bootstrap.min.js" integrity="sha384-JjSmVgyd0p3pXB1rRibZUAYoIIy6OrQ6VrjIEaFf/nJGzIxFDsf4x0xIM+B07jRM" crossorigin="anonymous"></script>

<!--
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha3/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-KK94CHFLLe+nY2dmCWGMq91rCGa5gtU4mk92HdvYe+M/SXH301p5ILy+dN9+nJOZ" crossorigin="anonymous">
	<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha3/dist/js/bootstrap.bundle.min.js" integrity="sha384-ENjdO4Dr2bkBIFxQpeoTz1HIcje39Wm4jDKdf19U8gI4ddQ3GYNS7NTKfAdVQSZe" crossorigin="anonymous"></script>
 -->
    <title>Choose your options</title>
    <script>
        var dress_codes = {
            'Men': {
                'casual_dress': ['T-shirt', 'Jeans', 'Sneakers'],
                'smart_casual': ['Collared shirt', 'Chinos', 'Loafers'],
                'business_casual': ['Button-down shirt', 'Khakis', 'Dress shoes'],
                'jacket_required': ['Blazer', 'Dress pants', 'Dress shoes']
            },
            'Women': {
                'casual_dress': ['Blouse', 'Skirt', 'Sandals'],
                'smart_casual': ['Blouse', 'Tailored pants', 'Flats'],
                'business_casual': ['Dress', 'Blazer', 'Heels'],
                'jacket_required': ['Elegant dress', 'Sophisticated separates', 'Heels']
            }
        };

function updateGarmentOptions() {
    var gender = document.getElementById("gender").value;
    var dress_code = document.getElementById("dress_code").value;
    var garment_options = dress_codes[gender][dress_code];
    var garment_container = document.getElementById("garment_container");

    garment_container.innerHTML = "";
    for (var i = 0; i < garment_options.length; i++) {
        var garment_div = document.createElement("div");
        garment_div.classList.add("form-group", "garment-grid");  // Add the garment-grid class  // Add Bootstrap class
        garment_div.innerHTML = '<h3>' + garment_options[i] + '</h3>' +
        '<label for="color_' + garment_options[i] + '">Color:</label>' +
        '<select name="color_' + garment_options[i] + '" id="color_' + garment_options[i] + + '" class="form-control">' +
        '<option value="Red">Red</option>' +
        '<option value="Orange">Orange</option>' +
        '<option value="Yellow">Yellow</option>' +
        '<option value="Green">Green</option>' +
        '<option value="Blue">Blue</option>' +
        '<option value="Purple">Purple</option>' +
        '<option value="Pink">Pink</option>' +
        '<option value="Brown">Brown</option>' +
        '<option value="Grey">Grey</option>' +
        '<option value="Black">Black</option>' +
        '<option value="White">White</option>' +
        '</select><br>' +
        '<label for="pattern_' + garment_options[i] + '">Pattern:</label>' +
        '<select name="pattern_' + garment_options[i] + '" id="pattern_' + garment_options[i] + '">' +
        '<option value="Solid">Solid</option>' +
        '<option value="Stripes">Stripes</option>' +
        '<option value="Plaid">Plaid</option>' +
        '<option value="Floral">Floral</option>' +
        '<option value="Polka Dots">Polka Dots</option>' +
        '</select><br>' +
        '<label for="fit_' + garment_options[i] + '">Fit:</label>' +
        '<select name="fit_' + garment_options[i] + '" id="fit_' + garment_options[i] + '">' +
        '<option value="Slim">Slim</option>' +
        '<option value="Regular">Regular</option>' +
        '<option value="Relaxed">Relaxed</option>' +
        '<option value="Tailored">Tailored</option>' +
        '<option value="Loose">Loose</option>' +
        '</select>';

        garment_container.appendChild(garment_div);
    }
}

function displayPreferences(event) {
    var gender = document.getElementById("gender").value;
    var dress_code = document.getElementById("dress_code").value;
    var garment_container = document.getElementById("garment_container");
    var garment_elements = garment_container.children;

    var preferences_summary = document.getElementById("preferences_summary");
    var gender_summary = document.getElementById("gender_summary");
    var garment_summary = document.getElementById("garment_summary");

    gender_summary.textContent = gender + "'s Garments:";
    garment_summary.innerHTML = "";

    for (var i = 0; i < garment_elements.length; i++) {
        var garment_name = garment_elements[i].querySelector("h3").textContent;
        var color = garment_elements[i].querySelector("[name^='color_']").value;
        var pattern = garment_elements[i].querySelector("[name^='pattern_']").value;
        var fit = garment_elements[i].querySelector("[name^='fit_']").value;

        var li = document.createElement("li");
        li.textContent = garment_name + " (" + color + ") (" + pattern + ") (" + fit + ")";
        garment_summary.appendChild(li);
    }

    preferences_summary.style.display = "block";
}

window.onload = updateGarmentOptions;  // Call the function when the page loads
        </script>
    </head>
    <body>
    	<div class="container"> <!-- Add the container div to center content -->
    		<h1 class="text-center">The Dress Code for the restaurant is</h1>
    		<div class="center-container">
    			<span id="dress_code_result" class="text-center d-block">
          {{ dress_code }}
    			</span>


			</div>

        	<h2 class="text-center">Choose your options</h2> <!-- Add the text-center class to center the title -->
        	<form action="/result" method="POST" class="text-left">

                <label for="gender">Gender:</label>
               	<select name="gender" id="gender" class="form-control" onchange="updateGarmentOptions()">
               	 	<option selected>Gender</option>
  					<option value="Men">Men</option>
  					<option value="Women">Women</option>
  					<option value="Prefer not to say">Prefer not to say</option>
				</select>
				<br>


                <label for="dress_code">Dress Code:</label>
                <select name="dress_code" id="dress_code" class="form-control" onchange="updateGarmentOptions()">
                	<option value="casual_dress">Casual Dress</option>
                	<option value="smart_casual">Smart Casual</option>
                	<option value="business_casual">Business Casual</option>
                	<option value="jacket_required">Jacket Required</option>
            	</select>
            	<br><br>


            	<div id="garment_container" class="garment-grid"> <!-- Add the garment-grid class -->
                <!-- The garment options will be displayed here -->
            	</div>
            	<br>
            	<div class="center-btn">
    				<input type="submit" class="btn btn-primary" value="Submit"> <!-- Add the "btn" and "btn-primary" classes to style the submit button -->
				</div>
        	</form>
    	</div>
	</body>
</html>
'''
# index_html = '''
# <!DOCTYPE html>
# <html lang="en">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Choose your options</title>
#     <script>
#         var dress_codes = {
#             'Men': {
#                 'casual_dress': ['T-shirt', 'Jeans', 'Sneakers'],
#                 'smart_casual': ['Collared shirt', 'Chinos', 'Loafers'],
#                 'business_casual': ['Button-down shirt', 'Khakis', 'Dress shoes'],
#                 'jacket_required': ['Blazer', 'Dress pants', 'Dress shoes']
#             },
#             'Women': {
#                 'casual_dress': ['Blouse', 'Skirt', 'Sandals'],
#                 'smart_casual': ['Blouse', 'Tailored pants', 'Flats'],
#                 'business_casual': ['Dress', 'Blazer', 'Heels'],
#                 'jacket_required': ['Elegant dress', 'Sophisticated separates', 'Heels']
#             }
#         };

#       function updateGarmentOptions() {
#     var gender = document.getElementById("gender").value;
#     var dress_code = document.getElementById("dress_code").value;
#     var garment_options = dress_codes[gender][dress_code];
#     var garment_container = document.getElementById("garment_container");

#     garment_container.innerHTML = "";
#     for (var i = 0; i < garment_options.length; i++) {
#         var garment_div = document.createElement("div");
#         garment_div.innerHTML = '<h3>' + garment_options[i] + '</h3>' +
#         '<label for="color_' + garment_options[i] + '">Color:</label>' +
#         '<select name="color_' + garment_options[i] + '" id="color_' + garment_options[i] + '">' +
#         '<option value="Red">Red</option>' +
#         '<option value="Orange">Orange</option>' +
#         '<option value="Yellow">Yellow</option>' +
#         '<option value="Green">Green</option>' +
#         '<option value="Blue">Blue</option>' +
#         '<option value="Purple">Purple</option>' +
#         '<option value="Pink">Pink</option>' +
#         '<option value="Brown">Brown</option>' +
#         '<option value="Grey">Grey</option>' +
#         '<option value="Black">Black</option>' +
#         '<option value="White">White</option>' +
#         '</select><br>' +
#         '<label for="pattern_' + garment_options[i] + '">Pattern:</label>' +
#         '<select name="pattern_' + garment_options[i] + '" id="pattern_' + garment_options[i] + '">' +
#         '<option value="Solid">Solid</option>' +
#         '<option value="Stripes">Stripes</option>' +
#         '<option value="Plaid">Plaid</option>' +
#         '<option value="Floral">Floral</option>' +
#         '<option value="Polka Dots">Polka Dots</option>' +
#         '</select><br>' +
#         '<label for="fit_' + garment_options[i] + '">Fit:</label>' +
#         '<select name="fit_' + garment_options[i] + '" id="fit_' + garment_options[i] + '">' +
#         '<option value="Slim">Slim</option>' +
#         '<option value="Regular">Regular</option>' +
#         '<option value="Relaxed">Relaxed</option>' +
#         '<option value="Tailored">Tailored</option>' +
#         '<option value="Loose">Loose</option>' +
#         '</select>';

#         garment_container.appendChild(garment_div);
#     }
# }

# function displayPreferences(event) {
#     var gender = document.getElementById("gender").value;
#     var dress_code = document.getElementById("dress_code").value;
#     var garment_container = document.getElementById("garment_container");
#     var garment_elements = garment_container.children;

#     var preferences_summary = document.getElementById("preferences_summary");
#     var gender_summary = document.getElementById("gender_summary");
#     var garment_summary = document.getElementById("garment_summary");

#     gender_summary.textContent = gender + "'s Garments:";
#     garment_summary.innerHTML = "";

#     for (var i = 0; i < garment_elements.length; i++) {
#         var garment_name = garment_elements[i].querySelector("h3").textContent;
#         var color = garment_elements[i].querySelector("[name^='color_']").value;
#         var pattern = garment_elements[i].querySelector("[name^='pattern_']").value;
#         var fit = garment_elements[i].querySelector("[name^='fit_']").value;

#         var li = document.createElement("li");
#         li.textContent = garment_name + " (" + color + ") (" + pattern + ") (" + fit + ")";
#         garment_summary.appendChild(li);
#     }

#     preferences_summary.style.display = "block";
# }


#         </script>
#     </head>
#     <body>
#         <h1>Choose your options</h1>
#         <form action="/result" method="POST">
#             <label for="gender">Gender:</label>
#             <select name="gender" id="gender" onchange="updateGarmentOptions()">
#                 <option value="Men">Men</option>
#                 <option value="Women">Women</option>
#             </select>
#             <br>
#             <label for="dress_code">Dress Code:</label>
#             <select name="dress_code" id="dress_code" onchange="updateGarmentOptions()">
#                 <option value="casual_dress">Casual Dress</option>
#                 <option value="smart_casual">Smart Casual</option>
#                 <option value="business_casual">Business Casual</option>
#                 <option value="jacket_required">Jacket Required</option>
#             </select>
#             <br><br>
#             <div id="garment_container">
#                 <!-- The garment options will be displayed here -->
#             </div>
#             <br>
#             <input type="submit" value="Submit">

#         </form>

#     </body>
# </html>
# '''

### Result Page

In [ ]:
result_html = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Result</title>
    <link href="https://stackpath.bootstrapcdn.com/bootstrap/4.3.1/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {background-color: #F6F6F6;}
        .jumbotron {background-color: #D9CAB3; color: white;}
        .card {background-color: #6D9886;}
        .btn-primary {background-color: #6D9886; border-color: 212121;}
        .btn_secondary {background-color: #6D9886; border-color: 212121;}
    </style>
</head>
<body class="container">
    <div class="jumbotron mt-4">
        <h1 class="display-4">Your Choices</h1>
        {% for garment, text in user_preferences.items() %}
            <div class="card mt-4">
                <div class="card-body">
                    <h5 class="card-title">{{ garment }}: {{ text }}</h5>
                    <img src="{{ image_data[text] }}" class="card-img-top img-thumbnail" style="max-width: 200px;" alt="{{ garment }}">
                </div>
            </div>
        {% endfor %}
    </div>
    <a href="{{ url_for('index') }}" class="btn btn-primary">Back to selection page</a>
    <a href="{{ url_for('dresscode') }}" class="btn btn-primary">Back to restaurant page</a>


</body>
</html>
'''

#    <a href="{{ url_for('dress_code') }}" class="btn btn-secondary">Back to restaurant page</a>

#css
# result_html = '''
# <!DOCTYPE html>
# <html lang="en">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Result</title>
#     <style>
#         body {
#             background-color: #f8f9fa;
#             color: #343a40;
#             font-family: Arial, sans-serif;
#         }
#         h1 {
#             text-align: center;
#             margin-top: 40px;
#         }
#         h3 {
#             margin-left: 40px;
#         }
#         img {
#             display: block;
#             margin-left: auto;
#             margin-right: auto;
#             width: 50%;
#             padding: 20px;
#         }
#         a {
#             display: block;
#             width: 200px;
#             height: 50px;
#             margin: 20px auto;
#             background-color: #343a40;
#             color: #ffffff;
#             text-align: center;
#             line-height: 50px;
#             text-decoration: none;
#             border-radius: 10px;
#         }
#         a:hover {
#             background-color: #007bff;
#         }
#     </style>
# </head>
# <body>
#     <h1>Your Choices</h1>
#     {% for garment, text in user_preferences.items() %}
#         <h3>{{ garment }}: {{ text }}</h3>
#         <img src="{{ image_data[text] }}" alt="{{ garment }}">
#     {% endfor %}
#     <a href="{{ url_for('index') }}">Back to selection page</a>
# </body>
# </html>
# '''

#image
# result_html = '''
# <!DOCTYPE html>
# <html lang="en">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Result</title>
# </head>
# <body>
#     <h1>Your Choices</h1>
#     {% for garment, text in user_preferences.items() %}
#         <h3>{{ garment }}: {{ text }}</h3>
#         <img src="{{ image_data[text] }}" alt="{{ garment }}">
#     {% endfor %}
#     <a href="{{ url_for('index') }}">Back to selection page</a>
# </body>
# </html>
# '''

#text
# '''
# <!DOCTYPE html>
# <html lang="en">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Result</title>
# </head>
# <body>
#     <h1>Your Choices</h1>
#     <p>Users' Preference: {{ user_preferences }}</p>
#     <a href="{{ url_for('index') }}">Back to selection page</a>
# </body>
# </html>
# '''

## Others

In [ ]:
with open('flask_app/templates/index.html', 'w') as f:
    f.write(index_html)

with open('flask_app/templates/result.html', 'w') as f:
    f.write(result_html)

with open('flask_app/templates/dress_code.html', 'w') as f:
    f.write(dress_code_html)


In [ ]:
import base64
import os

def image_path_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode('utf-8')
    return f"data:image/jpeg;base64,{encoded_image}"


In [ ]:
# @app.route('/dresscode', methods=['GET', 'POST'])
# def dresscode():
#     if request.method == 'POST':
#         # Process form data
#         user_choices = request.form.to_dict()
#         user_preferences = generate_user_preferences(user_choices)

#         # Redirect to index page with user preferences as query parameter
#         return redirect(url_for('index', user_preferences=user_preferences))

#     # If the request method is GET, render the dress_code template as usual
#     return render_template('dress_code.html')

# @app.route('/index', methods=['GET', 'POST'])
# def index():
#     # Get user preferences from query parameter, if it exists
#     user_preferences = request.args.get('user_preferences')
#     if user_preferences:
#         # Convert user preferences string to dictionary
#         user_preferences = ast.literal_eval(user_preferences)

#     return render_template('index.html', user_preferences=user_preferences)

In [ ]:
from flask import Flask, render_template, request, send_from_directory,url_for, redirect
from flask_ngrok import run_with_ngrok


app = Flask(__name__, template_folder='flask_app/templates')
run_with_ngrok(app)


def generate_user_preferences(user_choices):
    garments = {}

    # Extract unique garment types from user_choices keys
    for key in user_choices.keys():
        if key != 'gender' and key != 'dress_code':
            garment_key = key.split('_')[1]
            if garment_key not in garments:
                garments[garment_key] = {'color': '', 'pattern': '', 'fit': ''}

    # Assign attribute values to each garment
    for key, value in user_choices.items():
        if key != 'gender' and key != 'dress_code':
            garment_key = key.split('_')[1]
            attribute = key.split('_')[0]
            garments[garment_key][attribute] = value

    # Create a summary dictionary with garment descriptions
    summary = {}
    for garment_key, attributes in garments.items():
        summary[garment_key] = f"{attributes['color']} {attributes['pattern']} {attributes['fit']} {garment_key}"

    return summary
# @app.route('/', methods=['GET', 'POST'])
# def dress_code():
#     restaurant_name = ""
#     dress_code = ""

#     if request.method == 'POST':
#         # Get the restaurant name from the user input
#         restaurant_name = request.form['restaurant_name']

#         # Call the method to get the dress code based on the restaurant name
#         dress_code = get_dress_code(restaurant_name)

#     # Render the dress_code.html template with the restaurant name and dress code (if provided)
#     return render_template('dress_code.html', restaurant_name=restaurant_name, dress_code=dress_code)

@app.route('/', methods=['GET', 'POST'])
def dresscode():
    if request.method == 'POST':
        restaurant = request.form.get('restaurant')
        if restaurant:
            dress_code = get_dress_code(restaurant)  # Assuming get_dress_code() is defined
            return redirect(url_for('index', dress_code=dress_code))
    return render_template('dress_code.html')


@app.route('/index', methods=['GET','POST'])
def index():
    dress_code = request.args.get('dress_code', "Not Found")
    return render_template('index.html', dress_code=dress_code)
# @app.route('/', methods=['GET', 'POST'])
# def dresscode():
#     return render_template('dress_code.html')

# @app.route('/index', methods=['GET', 'POST'])
# def index():


#     return render_template('index.html')

@app.route('/result', methods=['POST'])
def result():
    user_choices = request.form.to_dict()
    user_preferences = generate_user_preferences(user_choices)

    # Find image paths and convert them to base64
    image_data = {}
    for garment, text in user_preferences.items():
        try:
            image_path = find_image_path(text)
            image_data[text] = image_path_to_base64(image_path)
        except Exception as e:
            print(f"Error finding image path for {text}: {e}")
            image_data[text] = None

    return render_template('result.html', user_preferences=user_preferences, image_data=image_data)


#version shows image path, don't delete
# @app.route('/')
# def index():
#     return render_template('index.html')

# @app.route('/result', methods=['POST'])
# def result():
#     user_choices = request.form.to_dict()
#     user_preferences = generate_user_preferences(user_choices)

#     # Find image paths
#     image_paths = {}
#     for garment, text in user_preferences.items():
#         image_path = find_image_path(text)
#         image_paths[text] = image_path

#     return render_template('result.html', user_preferences=user_preferences, image_paths=image_paths)

#this version shows text only, but don't delete plz!!!
# @app.route('/')
# def index():
#     return render_template('index.html')

# @app.route('/result', methods=['POST'])
# def result():
#     user_choices = request.form.to_dict()
#     user_preferences = generate_user_preferences(user_choices)

#     return render_template('result.html', user_preferences=user_preferences)


app.run()